<a href="https://colab.research.google.com/github/expaetra/CM3070_final_project/blob/master/08_data_expansion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/expaetra/CM3070_final_project.git
%cd CM3070_final_project

fatal: destination path 'CM3070_final_project' already exists and is not an empty directory.
/content/CM3070_final_project


In [2]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import time
from urllib.parse import quote

This notebook extends the data to improve the classification process. The dataset is collected in a balanced way across time. For each category the notebook retrieeves up to 500 papers for each year from 2021 to 2025. This makes the dataset more suitable for comparison across categories and years than simply collecting the most recent papers.

This notebook containst the scraper and the necessariy cleaning steps for the dataset to be ready for classification.

### Scraper

In [3]:
taxonomy = {
    'cs.LG': {'field': 'Machine Learning', 'discipline': 'Artificial Intelligence'},
    'cs.CV': {'field': 'Computer Vision', 'discipline': 'Computer Imaging and Vision'},
    'cs.AI': {'field': 'General Artificial Intelligence', 'discipline': 'Artificial Intelligence'},
    'cs.CL': {'field': 'Natural Language Processing', 'discipline': 'Artificial Intelligence'},
    'cs.IT': {'field': 'Information Theory', 'discipline': 'Communication'},
    'cs.RO': {'field': 'General Robotics', 'discipline': 'Robotics'},
    'cs.CR': {'field': 'Cryptography', 'discipline': 'Cybersecurity'},
    'cs.HC': {'field': 'General Human-Computer Interaction', 'discipline': 'Human-Computer Interaction'},
    'cs.DS': {'field': 'Data Structures and Algorithms', 'discipline': 'Theory of Computation'},
    'cs.DC': {'field': 'Distributed Computer Systems', 'discipline': 'Distributed Systems'},
    'cs.CY': {'field': 'Computers and Society', 'discipline': 'Social Computing'},
    'cs.NI': {'field': 'General Computer Networks', 'discipline': 'Computer Networks'},
    'cs.SE': {'field': 'General Software Engineering', 'discipline': 'Software Engineering'},
    'cs.IR': {'field': 'General Information Retrieval', 'discipline': 'Information Retrieval'},
    'cs.SI': {'field': 'Social Networks', 'discipline': 'World Wide Web'},
    'cs.SD': {'field': 'Sound and Signal Processing', 'discipline': 'Multimedia'},
    'cs.LO': {'field': 'Formal Logic', 'discipline': 'Artificial Intelligence'},
    'cs.NE': {'field': 'Neural Networks', 'discipline': 'Artificial Intelligence'},
    'cs.DM': {'field': 'Discrete Mathematics', 'discipline': 'Theory of Computation'},
    'cs.GT': {'field': 'Game Theory', 'discipline': 'Artificial Intelligence'},
    'cs.CC': {'field': 'Computational Complexity', 'discipline': 'Theory of Computation'},
    'cs.MA': {'field': 'Multiagent Systems', 'discipline': 'Artificial Intelligence'},
    'cs.DB': {'field': 'Database Systems', 'discipline': 'Database Systems'},
    'cs.CE': {'field': 'Computational Engineering', 'discipline': 'Computational Science'},
    'cs.PL': {'field': 'Programming Languages', 'discipline': 'Computer Programming'},
    'cs.MM': {'field': 'General Multimedia', 'discipline': 'Multimedia'},
    'cs.CG': {'field': 'Computational Geometry', 'discipline': 'Computer Graphics'},
    'cs.AR': {'field': 'Hardware Architecture', 'discipline': 'Computer Hardware'},
    'cs.ET': {'field': 'Emerging Technologies', 'discipline': 'Emerging Technologies'},
    'cs.DL': {'field': 'Digital Libraries', 'discipline': 'World Wide Web'},
    'cs.FL': {'field': 'Formal Languages and Automata Theory', 'discipline': 'Theoretical Computer Science'},
    'cs.PF': {'field': 'Performance', 'discipline': 'Computer Systems'},
}

In [4]:
#  years included in the scraping process

years = [2021, 2022, 2023, 2024, 2025]
years

[2021, 2022, 2023, 2024, 2025]

In [5]:
# collect the abstracts

def scrape_arxiv(category, year, max_results=500, wait=3):

    # initialize a list to store abstracts
    abstracts =[]
    batch_size = 100
    start = 0

    # define start range for the year
    start_date = f"{year}01010000"
    end_date = f"{year}12312359"

    # loop and collect abstracts
    while len(abstracts) < max_results:
        current_batch_size = min(batch_size, max_results-len(abstracts))

        # set API qquery parameters
        params = {
            "search_query": f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]",
            "start": start,
            "max_results": current_batch_size,
            "sortBy": "submittedDate",
            "sortOrder": "descending"
        }

        try:
            # send request to arxiv API
            response =requests.get(
                "https://export.arxiv.org/api/query",
                params=params,
                timeout = 30
            )

            # parse the response
            root = ET.fromstring(response.content)
        except Exception as e:
            print(f"Error on {category}, {year} at offset {start}: {e}, retrying in 30s...")
            time.sleep(30)
            continue

        # extract the papers
        entries=root.findall('{http://www.w3.org/2005/Atom}entry')

        if not entries:
            break

        # colelct the abstracts form the papers
        for entry in entries:
            abstract = entry.find('{http://www.w3.org/2005/Atom}summary')
            if abstract is not None:
                abstracts.append(abstract.text.strip())

            if len(abstracts) >= max_results:
                break

        # move to the next batch
        start +=current_batch_size
        time.sleep(wait)

    return abstracts

In [6]:
# empty list for the dataset

records = []

In [7]:
# collection proccess

for category, info in taxonomy.items():
    print(f"Scraping: {category}")

    for year in years:
        print(f"  Year: {year}")

        abstracts = scrape_arxiv(category, year, max_results = 500)
        for abstract in abstracts:
            records.append({
                'category': category,
                'field': info['field'],
                'discipline': info['discipline'],
                'year': year,
                'abstract': abstract
            })

        print(f"    {len(abstracts)} abstracts collected!")

print(f"\nTotal: {len(records)}")

Scraping: cs.LG
  Year: 2021
    500 abstracts collected!
  Year: 2022
    500 abstracts collected!
  Year: 2023
    500 abstracts collected!
  Year: 2024
    500 abstracts collected!
  Year: 2025
    500 abstracts collected!
Scraping: cs.CV
  Year: 2021
    500 abstracts collected!
  Year: 2022
    500 abstracts collected!
  Year: 2023
    500 abstracts collected!
  Year: 2024
    500 abstracts collected!
  Year: 2025
    500 abstracts collected!
Scraping: cs.AI
  Year: 2021
    500 abstracts collected!
  Year: 2022
    500 abstracts collected!
  Year: 2023
    500 abstracts collected!
  Year: 2024
    500 abstracts collected!
  Year: 2025
    500 abstracts collected!
Scraping: cs.CL
  Year: 2021
    500 abstracts collected!
  Year: 2022
    500 abstracts collected!
  Year: 2023
    500 abstracts collected!
  Year: 2024
    500 abstracts collected!
  Year: 2025
    500 abstracts collected!
Scraping: cs.IT
  Year: 2021
    500 abstracts collected!
  Year: 2022
    500 abstracts collect

### Cleaning

In [8]:
# convert to dataframe and examine

df = pd.DataFrame(records)
df.head()
df.shape

(79216, 5)

In [9]:
df.groupby(["category", "year"]).size().reset_index(name="count")
df.groupby(["field", "year"]).size().reset_index(name= "count")

,field,year,count
0,Computational Complexity,2021,500
1,Computational Complexity,2022,500
2,Computational Complexity,2023,500
3,Computational Complexity,2024,500
4,Computational Complexity,2025,500
...,...,...,...
155,Sound and Signal Processing,2021,500
156,Sound and Signal Processing,2022,500
157,Sound and Signal Processing,2023,500
158,Sound and Signal Processing,2024,500


In [10]:
# check for duplicates

print(df.isnull().sum())
print(f"\nDuplicates: {df.duplicated().sum()}")

category      0
field         0
discipline    0
year          0
abstract      0
dtype: int64

Duplicates: 11


In [11]:
df = df.drop_duplicates()
print(f"Remaining records: {len(df)}")

Remaining records: 79205


In [12]:
# list unique disciplies

print(df['discipline'].unique())

['Artificial Intelligence' 'Computer Imaging and Vision' 'Communication'
 'Robotics' 'Cybersecurity' 'Human-Computer Interaction'
 'Theory of Computation' 'Distributed Systems' 'Social Computing'
 'Computer Networks' 'Software Engineering' 'Information Retrieval'
 'World Wide Web' 'Multimedia' 'Database Systems' 'Computational Science'
 'Computer Programming' 'Computer Graphics' 'Computer Hardware'
 'Emerging Technologies' 'Theoretical Computer Science' 'Computer Systems']


In [13]:
# clean text

df['abstract'] = df['abstract'].str.strip()

In [14]:
# clean text

df['abstract'] = df['abstract'].str.strip()

In [15]:
# abstract lenght & analysis

df['abstract_length'] = df['abstract'].str.split().str.len()

print(df['abstract_length'].describe())

count    79205.000000
mean       174.363437
std         53.087885
min          4.000000
25%        139.000000
50%        174.000000
75%        211.000000
max        496.000000
Name: abstract_length, dtype: float64


In [16]:
# print random abstracts

print("Random abstracts: \n")
for a in df['abstract'].sample(10, random_state=42):
    print(a[:300], "\n")

Random abstracts: 

In this work, we introduce the problem of scenic routes among points in Rd. The key development is the nature of the problem in terms of both defining the concept of scenic points and scenic routes and then coming up with algorithms that meet different criteria for the generated scenic routes. The s 

This dissertation builds a compositional cyber-physical systems theory to develop concrete semantics relating the above diverse views necessary for safety and security assurance. In this sense, composition can take two forms. The first is composing larger models from smaller ones within each individ 

Regular Path Queries (RPQs), which are essentially regular expressions to be matched against the labels of paths in labeled graphs, are at the core of graph database query languages like SPARQL. A way to solve RPQs is to translate them into a sequence of operations on the adjacency matrices of each  

Despite being responsible for state-of-the-art results in several compu

In [17]:
# remove irrelevant abstracts

noise_keywords = ['withdrawn', 'this paper has been withdrawn', 'this volume contains']
mask = df['abstract'].str.contains('|'.join(noise_keywords), case=False, na=False)
print(f"Irrelevant abstracts: {mask.sum()}")

# remove noise
df = df[~mask]

print(f"Remaining abstracts: {len(df)}")

Irrelevant abstracts: 74
Remaining abstracts: 79131


In [18]:
# remove very short abstracts

df = df[df['abstract_length'] >= 50]

print(f"Remaining abstracts: {len(df)}")

Remaining abstracts: 78411


In [19]:
# lowercase text

df['abstract'] = df['abstract'].str.lower()

In [20]:
# check for empty strings

print((df['abstract'].str.strip() == "").sum())

0


In [21]:
print(df['category'].value_counts())

category
cs.AR    2498
cs.SI    2496
cs.SD    2495
cs.CV    2495
cs.CL    2494
cs.LG    2494
cs.MM    2493
cs.DC    2493
cs.IR    2492
cs.CE    2492
cs.RO    2492
cs.HC    2491
cs.AI    2491
cs.NE    2490
cs.DB    2487
cs.MA    2487
cs.CY    2486
cs.SE    2484
cs.NI    2481
cs.CR    2480
cs.GT    2476
cs.DS    2468
cs.IT    2465
cs.PL    2456
cs.ET    2456
cs.CG    2443
cs.CC    2426
cs.DM    2413
cs.LO    2382
cs.DL    2231
cs.FL    2216
cs.PF    2168
Name: count, dtype: int64


In [22]:
print("Random abstracts: \n")
for a in df['abstract'].sample(10, random_state=42):
    print(a[:300], "\n")

Random abstracts: 

faults occurring in ad-hoc robot networks may fatally perturb their topologies leading to disconnection of subsets of those networks. optimal topology synthesis is generally resource-intensive and time-consuming to be done in real time for large ad-hoc robot networks. one should only perform topolog 

in this work we design graph neural network architectures that capture optimal approximation algorithms for a large class of combinatorial optimization problems, using powerful algorithmic tools from semidefinite programming (sdp). concretely, we prove that polynomial-sized message-passing algorithm 

we revisit the join ordering problem in query optimization. the standard exact algorithm, dpccp, has a worst-case running time of $o(3^n)$. this is prohibitively expensive for large queries, which are not that uncommon anymore. we develop a new algorithmic framework based on subset convolution. dpco 

the recurrence rebuild and retrieval method (r3m) is proposed in this p

In [32]:
# count per discipline
print("Discipline counts:\n")
print(df['discipline'].value_counts())

# count per field
print("Field counts:\n")
print(df['field'].value_counts())

Discipline counts:

discipline
Artificial Intelligence         17314
Theory of Computation            7307
Multimedia                       4988
World Wide Web                   4727
Computer Hardware                2498
Computer Imaging and Vision      2495
Distributed Systems              2493
Information Retrieval            2492
Computational Science            2492
Robotics                         2492
Human-Computer Interaction       2491
Database Systems                 2487
Social Computing                 2486
Software Engineering             2484
Computer Networks                2481
Cybersecurity                    2480
Communication                    2465
Computer Programming             2456
Emerging Technologies            2456
Computer Graphics                2443
Theoretical Computer Science     2216
Computer Systems                 2168
Name: count, dtype: int64
Field counts:

field
Hardware Architecture                   2498
Social Networks                         2

In [36]:
# save
import os

os.makedirs("/content/drive/MyDrive/CM3070_final_project/backend/data", exist_ok=True)
df.to_csv(
    "/content/drive/MyDrive/CM3070_final_project/backend/data/data_expansion.csv",
    index=False
)

print("Saved")

Saved
